In [26]:
import os
import glob
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [27]:
df = pd.read_csv("HAM10000_metadata.csv")

all_images = glob.glob("archive/*/*")  # finds archive/folder1/img.jpg and archive/folder2/img.jpg

image_map = {os.path.splitext(os.path.basename(path))[0]: path for path in all_images}

df["filepath"] = df["image_id"].map(image_map)

df = df.dropna(subset=["filepath"])

print("Total matched images:", len(df))

Total matched images: 10015


In [28]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["dx"])

print("Classes:", le.classes_)

# Split into train/val sets
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label_encoded"], random_state=42
)

print("Training samples:", len(train_df), "Validation samples:", len(val_df))

Classes: ['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']
Training samples: 8012 Validation samples: 2003


In [29]:
img_size = (224, 224)

def process_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, img_size)
    img = img / 255.0
    return img, label

In [30]:
train_ds = tf.data.Dataset.from_tensor_slices((train_df["filepath"].values, train_df["label_encoded"].values))
train_ds = train_ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

In [31]:
val_ds = tf.data.Dataset.from_tensor_slices((val_df["filepath"].values, val_df["label_encoded"].values))
val_ds = val_ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

In [32]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze pretrained layers

num_classes = df["label_encoded"].nunique()

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ ?                      │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ ?                      │   0 (unbuilt) │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [33]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 228s 889ms/step - accuracy: 0.6328 - loss: 1.3075 - val_accuracy: 0.6695 - val_loss: 1.1277
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 240s 953ms/step - accuracy: 0.6767 - loss: 1.1663 - val_accuracy: 0.6695 - val_loss: 1.1266
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 225s 893ms/step - accuracy: 0.6671 - loss: 1.1702 - val_accuracy: 0.6695 - val_loss: 1.1180
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 321s 1s/step - accuracy: 0.6757 - loss: 1.1484 - val_accuracy: 0.6695 - val_loss: 1.1359
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 231s 921ms/step - accuracy: 0.6757 - loss: 1.1429 - val_accuracy: 0.6695 - val_loss: 1.1405
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 238s 945ms/step - accuracy: 0.6682 - loss: 1.1568 - val_accuracy: 0.6695 - val_loss: 1.1095
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 229s 909ms/step - accuracy: 0.6774 - loss: 1.1275 - val_accuracy: 0.6695 - val_loss: 1.1015
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.6745 - loss: